# `data_utils.py` - Dataset Classes & Data Loaders

### Purpose
Provides all PyTorch `Dataset` and `DataLoader` components needed to train and evaluate the multi-aspect sentiment model.

### Key Concepts

### One sample = (text, aspect) pair
A single CSV row with N aspect labels becomes N training samples. For example, if a review has scores for `colour`, `smell`, and `price`, that becomes 3 separate samples in the dataset:
```
('Great shade, smells weird, overpriced', 'colour', label=positive)
('Great shade, smells weird, overpriced', 'smell',  label=negative)
('Great shade, smells weird, overpriced', 'price',  label=negative)
```

### Dependency Parsing 
`DependencyParsingDataset` extends the base dataset by pre-computing the spaCy dependency parse tree for each unique text. These trees are passed to the GCN as `edge_index` tensors.

## Classes & Functions

| Name | Type | Description |
|------|------|-------------|
| `CosmeticReviewDataset` | Dataset | Base dataset — tokenises text, returns (input_ids, aspect_id, label) |
| `DependencyParsingDataset` | Dataset | Extends base — adds pre-computed dependency parse trees |
| `DependencyParser` | class | spaCy wrapper for parsing text into dependency edge_index |
| `collate_fn_with_dependencies` | function | Custom batch collator that handles variable-size edge_index lists |
| `create_dataloaders()` | function | One-call factory to build train/val/test loaders |
| `compute_class_weights()` | function | Returns per-aspect class counts for initialising HybridLoss |

In [ ]:
import time
_start_time = time.time()
print("Starting: Loading dependencies and libraries...")

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer
from collections import Counter
from tqdm import tqdm

print(f"Done in {time.time() - _start_time:.2f}s")
print("Completed: Loading dependencies and libraries.")


## 1. Base Dataset - `CosmeticReviewDataset`
Loads a CSV file and expands each row into one sample per aspect. Tokenises the text with the RoBERTa tokeniser.

In [ ]:
print("Starting: Defining class CosmeticReviewDataset...")

class CosmeticReviewDataset(Dataset):
    """
    Dataset for cosmetic product reviews with multi-aspect sentiment labels.
    Each CSV row is expanded into one sample per labelled aspect column.
    """
    def __init__(self, data_path, tokenizer, config, aspect_names, is_train=True):
        split = "train" if is_train else "val/test"
        print(f"\n[Dataset] Loading {split} data from: {data_path}")

        self.data = pd.read_csv(data_path)
        print(f"[Dataset] Rows in CSV: {len(self.data)}")

        self.tokenizer    = tokenizer
        self.config       = config
        self.aspect_names = aspect_names
        self.is_train     = is_train

        data_config      = config["data"]
        self.max_length  = data_config["max_seq_length"]  # RoBERTa max is 512; using 128 for speed
        self.text_column = data_config["text_column"]     # Column name in the CSV that holds review text
        self.label_map   = config["aspects"]["label_map"] # {"negative": 0, "neutral": 1, "positive": 2}

        print(f"[Dataset] Text column: '{self.text_column}'  |  max_seq_length: {self.max_length}")
        print(f"[Dataset] Aspects ({len(aspect_names)}): {aspect_names}")
        print(f"[Dataset] Label map: {self.label_map}")
        print("[Dataset] Building (text, aspect, label) samples...")

        # Each CSV row becomes N samples — one per labelled aspect column.
        # A single review mentioning colour AND smell becomes two separate training samples.
        self.samples = self._prepare_samples()

        print(f"[Dataset] Total samples built: {len(self.samples)}  "
              f"(avg {len(self.samples)/len(self.data):.1f} samples per row)")
        self._print_statistics()

    def _prepare_samples(self):
        samples: list = []
        skipped_empty: int = 0

        for idx, row in tqdm(self.data.iterrows(), total=len(self.data), desc="  Expanding rows"):
            text = str(row[self.text_column]) if pd.notna(row[self.text_column]) else ""
            if not text.strip():
                skipped_empty += 1
                continue

            for aspect in self.aspect_names:
                # NaN in an aspect column means the review was not labelled for that aspect
                if pd.notna(row[aspect]):
                    label_str = str(row[aspect]).lower()
                    if label_str in self.label_map:  # Skip any malformed labels
                        samples.append({
                            "text"        : text,
                            "aspect"      : aspect,
                            "aspect_id"   : self.aspect_names.index(aspect),  # Integer index for embedding lookup
                            "label"       : self.label_map[label_str],         # Convert string to integer class
                            "original_idx": idx,  # Preserve CSV row index for MSR evaluation grouping
                        })

        if skipped_empty:
            print(f"  [Dataset] Skipped {skipped_empty} rows with empty text")
        return samples

    def _print_statistics(self):
        aspect_counts = Counter([s["aspect"] for s in self.samples])
        label_names   = {v: k for k, v in self.label_map.items()}
        label_counts  = Counter([s["label"]  for s in self.samples])

        print("[Dataset] Aspect distribution:")
        for aspect in self.aspect_names:
            print(f"  {aspect:<16}: {aspect_counts.get(aspect, 0)}")

        print("[Dataset] Label distribution:")
        for label_id in sorted(label_counts):
            print(f"  {label_names[label_id]:<10}: {label_counts[label_id]}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample   = self.samples[idx]
        encoding = self.tokenizer(
            sample["text"],
            add_special_tokens=True,   # Adds [CLS] at start and [SEP] at end
            max_length=self.max_length,
            padding="max_length",      # Pad shorter sequences so all batches are the same length
            truncation=True,           # Truncate sequences longer than max_length
            return_tensors="pt",       # Return PyTorch tensors
        )
        return {
            "input_ids"    : encoding["input_ids"].squeeze(0),        # (max_length,) — removes the batch dim added by tokenizer
            "attention_mask": encoding["attention_mask"].squeeze(0),  # (max_length,) — 0 for padding positions
            "aspect_id"    : torch.tensor(sample["aspect_id"], dtype=torch.long),
            "label"        : torch.tensor(sample["label"],     dtype=torch.long),
            "text"         : sample["text"],    # Raw string kept for LIME/SHAP explainability
            "aspect"       : sample["aspect"],  # Aspect name kept for per-aspect metric grouping
            "review_id"    : sample["original_idx"],  # CSV row index — used to group samples by review in MSR evaluation
        }

print("Completed: Defining class CosmeticReviewDataset.")


## 2. Dependency Parsing Dataset
Extends `CosmeticReviewDataset` with pre-computed spaCy dependency trees.

**Important edge pruning**: spaCy uses word-level indices, but RoBERTa uses subword (BPE) indices capped at `max_length`. We prune edges whose node indices exceed `max_length` to prevent `IndexError` in the GCN's `scatter_add_`.


In [ ]:
print("Starting: Defining class DependencyParsingDataset...")

class DependencyParsingDataset(CosmeticReviewDataset):
    """
    Extended dataset that includes dependency parsing information.
    Pre-computes spaCy parse trees once upfront to avoid repeated parsing during training.
    """
    def __init__(self, data_path, tokenizer, config, aspect_names,
                 dependency_parser=None, is_train=True):
        super().__init__(data_path, tokenizer, config, aspect_names, is_train)
        self.dependency_parser = dependency_parser

        if self.dependency_parser is not None:
            unique_texts = len(set(s["text"] for s in self.samples))
            print(f"[DepDataset] Pre-computing spaCy parse trees for {unique_texts} unique texts...")
            self.dependency_trees = self._compute_dependency_trees()
            print(f"[DepDataset] Parse trees ready ({len(self.dependency_trees)} unique texts)")
        else:
            print("[DepDataset] No dependency parser provided — edge_index will be empty")
            self.dependency_trees = None

    def _compute_dependency_trees(self):
        unique_texts = list(set(s["text"] for s in self.samples))
        trees: dict = {}
        parse_errors: list = []

        for text in tqdm(unique_texts, desc="  Parsing dependency trees"):
            try:
                toks, ei, et = self.dependency_parser.parse(text)
                trees[text]  = {"tokens": toks, "edge_index": ei, "edge_types": et}
            except Exception as e:
                parse_errors.append(str(e))
                trees[text] = {
                    "tokens"    : [],
                    "edge_index": torch.zeros((2, 0), dtype=torch.long),
                    "edge_types": [],
                }

        if parse_errors:
            print(f"  [DepDataset] WARNING: {len(parse_errors)} parse errors (set to empty edge_index)")
        return trees

    def __getitem__(self, idx):
        item = super().__getitem__(idx)  # Get standard (input_ids, label, ...) from parent

        if self.dependency_trees is not None:
            dep_info   = self.dependency_trees.get(item["text"], {})
            edge_index = dep_info.get("edge_index", torch.zeros((2, 0), dtype=torch.long))

            # spaCy uses word-level token indices, but the GCN operates over
            # RoBERTa's subword (BPE) token space capped at max_length.
            # Edges referencing positions beyond max_length would cause
            # an out-of-bounds error in scatter_add_, so we prune them here.
            if edge_index.size(1) > 0:
                mask       = (edge_index[0] < self.max_length) & (edge_index[1] < self.max_length)
                edge_index = edge_index[:, mask]

            item["edge_index"] = edge_index
            item["tokens"]     = dep_info.get("tokens", [])      # Token strings for XAI display
            item["edge_types"] = dep_info.get("edge_types", [])  # Dependency relation labels (e.g. "nsubj")
        return item

print("Completed: Defining class DependencyParsingDataset.")


## 3. Custom Collator
A standard DataLoader collator cannot handle lists of tensors with variable `edge_index` sizes. This custom function stacks fixed-size tensors normally and keeps dependency info as Python lists (one entry per sample in the batch).


In [ ]:
print("Starting: Defining function collate_fn_with_dependencies...")

def collate_fn_with_dependencies(batch):
    """Custom collate function for batches with dependency trees."""
    # edge_index tensors have a variable number of edges per sample, so they cannot
    # be stacked into a single tensor. We keep them as a Python list instead.
    edge_indices, tokens, edge_types = [], [], []
    for item in batch:
        if "edge_index" in item:
            edge_indices.append(item["edge_index"])
            tokens.append(item.get("tokens", []))
            edge_types.append(item.get("edge_types", []))
        else:
            # Samples from CosmeticReviewDataset (no dependency parsing) get None
            # so that the model's forward() can detect and skip the GCN branch.
            edge_indices.append(None)
            tokens.append([])
            edge_types.append([])

    return {
        # Fixed-size tensors can be stacked normally
        "input_ids"    : torch.stack([item["input_ids"]     for item in batch]),  # (B, seq_len)
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),  # (B, seq_len)
        "aspect_ids"   : torch.stack([item["aspect_id"]    for item in batch]),  # (B,)
        "labels"       : torch.stack([item["label"]         for item in batch]),  # (B,)
        # Variable-length / non-tensor fields stay as Python lists
        "review_ids"   : [item["review_id"] for item in batch],
        "texts"        : [item["text"]      for item in batch],
        "aspects"      : [item["aspect"]    for item in batch],
        "edge_indices" : edge_indices,   # List of (2, num_edges) tensors or None
        "tokens"       : tokens,
        "edge_types"   : edge_types,
    }

print("Completed: Defining function collate_fn_with_dependencies.")


## 4. DependencyParser - spaCy Wrapper
Wraps spaCy's `en_core_web_sm` English model to convert raw review text into a dependency `edge_index` tensor that the GCN can consume. If the model is not installed it downloads it automatically. The `language` parameter is reserved for future multilingual support.


In [ ]:
print("Starting: Defining class DependencyParser...")

class DependencyParser:
    """Wrapper for dependency parsing using spaCy."""

    def __init__(self, language="en", model_name="en_core_web_sm"):
        import spacy
        print(f"[DepParser] Loading spaCy model: {model_name}")
        try:
            self.nlp = spacy.load(model_name)
            print("[DepParser] Model loaded OK")
        except OSError:
            print(f"[DepParser] Model not found — downloading {model_name}...")
            import subprocess
            subprocess.run(["python", "-m", "spacy", "download", model_name])
            self.nlp = spacy.load(model_name)
            print("[DepParser] Model downloaded and loaded OK")

    def parse(self, text):
        """
        Returns:
            tokens:     list of token strings
            edge_index: (2, num_edges) LongTensor  [head -> dependent]
            edge_types: list of dependency relation strings (e.g. "nsubj", "dobj")
        """
        doc        = self.nlp(text)
        tokens     = [token.text for token in doc]
        edges      = []
        edge_types = []

        for token in doc:
            if token.head != token:  # Exclude self-loops (root node has head == itself)
                edges.append([token.head.i, token.i])  # Directed edge: head -> dependent
                edge_types.append(token.dep_)

        edge_index = (torch.tensor(edges, dtype=torch.long).t()
                      if edges else torch.zeros((2, 0), dtype=torch.long))
        return tokens, edge_index, edge_types

print("Completed: Defining class DependencyParser.")


## 5. Factory Functions
`create_dataloaders()` builds all three DataLoaders (train/val/test) in one call, picking `DependencyParsingDataset` or `CosmeticReviewDataset` automatically based on config.
`compute_class_weights()` returns per-aspect `[neg_count, neu_count, pos_count]` lists that `AspectSpecificLossManager` uses to initialise weighted loss functions.


In [ ]:
print("Starting: Defining factory functions...")

def create_dataloaders(config, tokenizer, dependency_parser=None):
    """Create train / val / test DataLoaders from config."""
    data_config  = config["data"]
    hw_config    = config["hardware"]
    aspect_names = config["aspects"]["names"]

    use_dep = data_config.get("use_dependency_parsing", False) and dependency_parser is not None
    DatasetClass = DependencyParsingDataset if use_dep else CosmeticReviewDataset
    extra = {"dependency_parser": dependency_parser} if use_dep else {}

    print(f"\n[DataLoaders] Dataset class: {DatasetClass.__name__}")
    print(f"[DataLoaders] Batch size: {config['training']['batch_size']}  |  "
          f"num_workers: {hw_config['num_workers']}")

    splits = [
        ("train", data_config["train_path"], True),
        ("val",   data_config["val_path"],   False),
        ("test",  data_config["test_path"],  False),
    ]
    loaders = []
    for name, path, shuffle in splits:
        print(f"\n[DataLoaders] Building {name} loader from: {path}")
        ds = DatasetClass(path, tokenizer, config, aspect_names, is_train=shuffle, **extra)
        loader = DataLoader(
            ds,
            batch_size=config["training"]["batch_size"],
            shuffle=shuffle,
            num_workers=hw_config["num_workers"],
            pin_memory=hw_config["pin_memory"],
            collate_fn=collate_fn_with_dependencies,
        )
        print(f"[DataLoaders] {name}: {len(ds)} samples -> {len(loader)} batches")
        loaders.append(loader)

    return tuple(loaders)


def compute_class_weights(data_path, aspect_names, label_map):
    """
    Returns {aspect: [neg_count, neu_count, pos_count]} for HybridLoss init.
    """
    print(f"\n[ClassWeights] Computing class counts from: {data_path}")
    df = pd.read_csv(data_path)
    aspect_class_counts = {}

    for aspect in tqdm(aspect_names, desc="  Counting per aspect"):
        # Counts are ordered by label_id (0=neg, 1=neu, 2=pos) to match HybridLoss's
        # samples_per_class argument format. Using .astype(str) handles mixed types.
        counts = [0, 0, 0]
        for label_str, label_id in label_map.items():
            counts[label_id] = int((df[aspect].astype(str) == label_str).sum())
        aspect_class_counts[aspect] = counts

    print("[ClassWeights] Done. Summary (neg / neu / pos):")
    for asp, c in aspect_class_counts.items():
        ratio = f"  pos:neg={c[2]}:{c[0]}" if c[0] > 0 else "  (no negatives)"
        print(f"  {asp:<16}: neg={c[0]:>5}  neu={c[1]:>5}  pos={c[2]:>5}{ratio}")

    return aspect_class_counts

print("Completed: Factory functions defined.")


## Quick Test

Verifies the dataset and dependency parser work without needing any CSV file - builds a tiny in-memory DataFrame and runs through tokenisation and parsing.

In [ ]:
import time, os, tempfile
_start = time.time()
print("Running data_utils quick test...")

# --- minimal config ---
config = {
    'data': {'max_seq_length': 32, 'text_column': 'data',
             'train_path': '', 'val_path': '', 'test_path': '',
             'use_dependency_parsing': False},
    'hardware': {'num_workers': 0, 'pin_memory': False},
    'training': {'batch_size': 4},
    'aspects': {
        'names': ['colour', 'smell', 'price'],
        'label_map': {'negative': 0, 'neutral': 1, 'positive': 2},
    },
}
aspect_names = config['aspects']['names']
label_map    = config['aspects']['label_map']

# --- write a tiny CSV to a temp file so CosmeticReviewDataset can load it ---
import pandas as pd
rows = [
    {'data': 'I love the colour but the smell is awful.',  'colour': 'positive', 'smell': 'negative', 'price': float('nan')},
    {'data': 'Great price, the texture is disappointing.', 'colour': float('nan'), 'smell': float('nan'), 'price': 'positive'},
    {'data': 'Neutral on everything honestly.',             'colour': 'neutral', 'smell': 'neutral', 'price': 'neutral'},
]
df = pd.DataFrame(rows)

tmp = tempfile.NamedTemporaryFile(suffix='.csv', delete=False, mode='w')
df.to_csv(tmp.name, index=False)
tmp.close()

# --- tokeniser ---
print("\nLoading RoBERTa tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# --- dataset ---
config['data']['train_path'] = tmp.name
dataset = CosmeticReviewDataset(tmp.name, tokenizer, config, aspect_names)

print(f"\nDataset has {len(dataset)} samples")
sample = dataset[0]
print(f"Sample keys:     {list(sample.keys())}")
print(f"input_ids shape: {sample['input_ids'].shape}")
print(f"aspect:          {sample['aspect']}")
print(f"label:           {sample['label'].item()}  ({['negative','neutral','positive'][sample['label'].item()]})")

# --- DataLoader (single batch) ---
from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size=len(dataset), collate_fn=collate_fn_with_dependencies)
batch  = next(iter(loader))
print(f"\nBatch input_ids shape:    {batch['input_ids'].shape}   (samples × seq_len)")
print(f"Batch aspect_ids:         {batch['aspect_ids'].tolist()}")
print(f"Batch labels:             {batch['labels'].tolist()}")

# --- DependencyParser ---
print("\nTesting DependencyParser...")
parser = DependencyParser()
text   = "The colour is beautiful but the smell is awful."
tokens, edge_index, edge_types = parser.parse(text)
print(f"Text:        '{text}'")
print(f"Tokens:      {tokens}")
print(f"Edge index:  shape {edge_index.shape}  ({edge_index.size(1)} edges)")
print(f"Edge types:  {edge_types[:5]}{'...' if len(edge_types) > 5 else ''}")

# --- compute_class_weights ---
weights = compute_class_weights(tmp.name, aspect_names, label_map)
print(f"\nClass counts per aspect:")
for asp, counts in weights.items():
    print(f"  {asp}: neg={counts[0]}, neu={counts[1]}, pos={counts[2]}")

os.unlink(tmp.name)  # clean up temp file
print(f"\nAll data_utils tests PASSED  ({time.time() - _start:.2f}s)")